# Synthetic endpoint and batch example

For the full multimodal workflow, start with the
[public-data notebook](public_multimodal_walkthrough.ipynb).

This optional example creates an endpoint and a separate batch job.
Deploy the subscribed SMB v2.0 Marketplace model package, predict OS and TTNT from synthetic EHR inputs, run Batch Transform, and delete the demo resources.

SMB v2.0 returns **rank-preserving pseudo-days**, not calibrated survival times or probabilities. Set MODEL_PACKAGE_ARN to the subscribed model-package ARN for your AWS region in the configuration cell below. Endpoint and batch cells create billable AWS resources; run the cleanup cell when finished.


In [ ]:
%pip install --quiet boto3 polars

In [ ]:
import io
import json
import math
import tarfile
import time
import uuid

import boto3
import polars as pl
from botocore.config import Config
from botocore.exceptions import ClientError

MODEL_PACKAGE_ARN = ""  # Released Marketplace model-package ARN after subscribing.
EXECUTION_ROLE_ARN = ""  # Your SageMaker execution role, with access to this S3 bucket.
BUCKET = ""  # Existing bucket in the same AWS region.
REGION = boto3.Session().region_name or "us-east-1"

assert MODEL_PACKAGE_ARN and EXECUTION_ROLE_ARN and BUCKET, (
    "Set package ARN, execution role and bucket."
)
assert MODEL_PACKAGE_ARN.split(":")[3] == REGION, (
    "Package and deployment region must match."
)
session = boto3.Session(region_name=REGION)
sm = session.client("sagemaker")
s3 = session.client("s3")
bucket_region = s3.get_bucket_location(Bucket=BUCKET).get("LocationConstraint")
bucket_region = {None: "us-east-1", "EU": "eu-west-1"}.get(bucket_region, bucket_region)
assert bucket_region == REGION, "S3 bucket and deployment region must match."
runtime = session.client(
    "sagemaker-runtime",
    config=Config(read_timeout=70, retries={"total_max_attempts": 1}),
)
package = sm.describe_model_package(ModelPackageName=MODEL_PACKAGE_ARN)
spec = package["InferenceSpecification"]
qualified = sorted(
    set(spec["SupportedRealtimeInferenceInstanceTypes"])
    & set(spec["SupportedTransformInstanceTypes"])
)
assert qualified, "Package must support the demonstrated endpoint and batch modes."
INSTANCE_TYPE = qualified[0]
run_id = "smb-os-ttnt-" + time.strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
prefix = "smb-marketplace-demo/" + run_id
model_name = endpoint_name = endpoint_config_name = run_id
transform_name = run_id + "-batch"
print(
    {
        "instance_type": INSTANCE_TYPE,
        "endpoint_name": endpoint_name,
        "s3_prefix": prefix,
    }
)

The synthetic patient has one observed lab at day 10 and a later lab at day 200. Both requests use a day-100 observation cutoff. The notebook constructs these inputs using synthetic data and public Python packages.


In [ ]:
tables = {
    "patient_characteristics.parquet": pl.DataFrame(
        {
            "PTID": ["synthetic-patient"],
            "VITAL_STATUS": ["Alive"],
            "LAST_ALIVE_INTERVAL": [300],
            "DEATH_INTERVAL": [None],
        }
    ),
    "labs.parquet": pl.DataFrame(
        {
            "TEST_INTERVAL": [10, 200],
            "CD_DESC": ["hemoglobin", "future"],
            "CD": ["x", "y"],
            "TEST_RESULT_VALUE": [12.0, 99.0],
            "TEST_RESULT_UOM": ["g/dL", "g/dL"],
        }
    ),
}


def make_payload(task_id):
    request = {
        "schema_version": "1.0",
        "request_uuid": str(uuid.uuid4()),
        "task_id": task_id,
        "task_version": "1.0",
        "task_inputs": {},
        "observation_cutoff_day_offset": 100,
        "data_objects": [
            {"modality": "ehr", "artifact_kind": "archive", "local_path": "/input/ehr"}
        ],
    }
    files = {"request.json": json.dumps(request).encode()}
    for name, frame in tables.items():
        stream = io.BytesIO()
        frame.write_parquet(stream)
        files["input/ehr/" + name] = stream.getvalue()
    output = io.BytesIO()
    with tarfile.open(fileobj=output, mode="w") as archive:
        for name, content in files.items():
            member = tarfile.TarInfo(name)
            member.size = len(content)
            member.mode = 0o600
            archive.addfile(member, io.BytesIO(content))
    return request, output.getvalue()


payloads = {
    "os": make_payload("time_to_death"),
    "ttnt": make_payload("time_to_next_treatment_or_death"),
}


def verify_prediction(value, request):
    assert value["schema_version"] == "1.0"
    for field in ("request_uuid", "task_id", "task_version"):
        assert value[field] == request[field]
    assert value["score_type"] == "rank_preserving_pseudo_days"
    prediction = value["prediction"]
    assert not isinstance(prediction, bool) and isinstance(prediction, (int, float))
    assert math.isfinite(prediction) and prediction > 0
    return value

Create an isolated SageMaker endpoint from the subscribed package. Configuration comes from the package and each request; no container environment overrides are needed.


In [ ]:
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=EXECUTION_ROLE_ARN,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    EnableNetworkIsolation=True,
)
sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InitialInstanceCount": 1,
            "InstanceType": INSTANCE_TYPE,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",
            "InitialVariantWeight": 1.0,
            "ContainerStartupHealthCheckTimeoutInSeconds": 900,
        }
    ],
)
sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)
print("Endpoint is in service")

In [ ]:
endpoint_predictions = {}
for label, (request, payload) in payloads.items():
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/x-tar",
        Accept="application/json",
        Body=payload,
    )
    with response["Body"] as stream:
        value = json.load(stream)
    endpoint_predictions[label] = verify_prediction(value, request)
endpoint_predictions

For Batch Transform, each S3 input object is one complete uncompressed tar. Upload only the tar objects to the input prefix. One concurrent transform matches the container's single GPU replica. MaxPayloadInMB=0 enables chunked delivery. For existing large patient files, use the archive layout and Batch Transform settings in README.md. This notebook keeps only its small synthetic fixtures in memory.


In [ ]:
for label, (_, payload) in payloads.items():
    s3.put_object(
        Bucket=BUCKET,
        Key=prefix + "/input/" + label + ".tar",
        Body=payload,
        ContentType="application/x-tar",
    )
sm.create_transform_job(
    TransformJobName=transform_name,
    ModelName=model_name,
    BatchStrategy="SingleRecord",
    MaxConcurrentTransforms=1,
    MaxPayloadInMB=0,
    ModelClientConfig={"InvocationsTimeoutInSeconds": 900, "InvocationsMaxRetries": 1},
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": "s3://" + BUCKET + "/" + prefix + "/input/",
            }
        },
        "ContentType": "application/x-tar",
        "CompressionType": "None",
        "SplitType": "None",
    },
    TransformOutput={
        "S3OutputPath": "s3://" + BUCKET + "/" + prefix + "/output/",
        "Accept": "application/json",
        "AssembleWith": "None",
    },
    TransformResources={
        "InstanceType": INSTANCE_TYPE,
        "InstanceCount": 1,
        "TransformAmiVersion": "al2-ami-sagemaker-batch-gpu-535",
    },
)
sm.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=transform_name
)
assert (
    sm.describe_transform_job(TransformJobName=transform_name)["TransformJobStatus"]
    == "Completed"
)

batch_predictions = {}
for page in s3.get_paginator("list_objects_v2").paginate(
    Bucket=BUCKET, Prefix=prefix + "/output/"
):
    for item in page.get("Contents", []):
        key = item["Key"]
        if not key.endswith(".out"):
            continue
        label = key.rsplit("/", 1)[-1].removesuffix(".tar.out")
        assert label in payloads
        with s3.get_object(Bucket=BUCKET, Key=key)["Body"] as stream:
            value = json.load(stream)
        batch_predictions[label] = verify_prediction(value, payloads[label][0])
        assert math.isclose(
            value["prediction"],
            endpoint_predictions[label]["prediction"],
            rel_tol=1e-6,
            abs_tol=1e-8,
        )
assert set(batch_predictions) == set(payloads)
batch_predictions

Run cleanup even if an earlier cell failed. It deletes only this demo's endpoint, endpoint configuration, model and S3 prefix. Completed transform-job records remain in SageMaker. Review any reported cleanup errors before closing the notebook.


In [ ]:
try:
    transform_status = sm.describe_transform_job(TransformJobName=transform_name)[
        "TransformJobStatus"
    ]
    if transform_status == "InProgress":
        sm.stop_transform_job(TransformJobName=transform_name)
    if transform_status in {"InProgress", "Stopping"}:
        sm.get_waiter("transform_job_completed_or_stopped").wait(
            TransformJobName=transform_name
        )
except ClientError as error:
    print("Transform cleanup:", error.response["Error"]["Code"])
try:
    sm.delete_endpoint(EndpointName=endpoint_name)
    sm.get_waiter("endpoint_deleted").wait(EndpointName=endpoint_name)
except ClientError as error:
    print("Endpoint cleanup:", error.response["Error"]["Code"])
for method, arguments in [
    (sm.delete_endpoint_config, {"EndpointConfigName": endpoint_config_name}),
    (sm.delete_model, {"ModelName": model_name}),
]:
    try:
        method(**arguments)
    except ClientError as error:
        print("Resource cleanup:", arguments, error.response["Error"]["Code"])
for page in s3.get_paginator("list_objects_v2").paginate(
    Bucket=BUCKET, Prefix=prefix + "/"
):
    objects = [{"Key": item["Key"]} for item in page.get("Contents", [])]
    if objects:
        result = s3.delete_objects(Bucket=BUCKET, Delete={"Objects": objects})
        assert not result.get("Errors"), result.get("Errors")
print("Cleanup requests completed; check any errors above.")